
# 🚗 Highway Planning Stack – Dynamic Bicycle Model, Lanelet A*, FSM, IDM, Frenet Lattice & Oriented Rectangle Collision

This notebook includes:

- **Map / Routing**
  - Lanelet-style straight highway
  - A* on lanelets for **global route** (lane sequence)
- **Vehicles & Dynamics**
  - Ego uses a **(simplified) dynamic bicycle model**:
    - states: x, y, yaw, v_x, v_y, yaw_rate
    - controls: longitudinal acceleration a_x (from IDM), steering angle δ (Stanley-like lane/trajectory tracking)
  - Other vehicles use lane-based **IDM car-following** (kinematic update)
- **Behavior Planning**
  - Route-aware FSM with lane-change locking
  - States: CRUISE, FOLLOW, STOP, LANE_CHANGE_LEFT, LANE_CHANGE_RIGHT, ACCELERATE
- **Local Planning (Frenet lattice – lateral)**
  - Candidates over multiple target lanes & time horizons (T ∈ {3, 4, 5} s)
  - Lateral quintic polynomials d(t)
  - Cost: jerk + collision risk + route deviation + mild T preference
  - All candidates visualized (gray) and chosen one (green)
- **Longitudinal (s/x)**
  - IDM for ego and all other vehicles
- **Collision**
  - Vehicles rendered as oriented rectangles (ego red, others blue)
  - Rectangle–rectangle collision using **Separating Axis Theorem (SAT)**; no false side-by-side collisions
- **Outputs**
  - Gradio app
  - GIF animation
  - CSV log with `t, x, y, lane_index, speed, behavior`


## 0. Imports & Global Constants

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import gradio as gr
from PIL import Image
import csv, os, tempfile, random

from matplotlib.patches import Rectangle

# Geometry
LANE_WIDTH = 3.7   # lane width [m]
VEHICLE_LENGTH = 4.5  # vehicle length [m]
VEHICLE_WIDTH  = 2.0  # vehicle width [m]

def lane_center_y(lane_index):
    """Lane 1 center is y=0, lane 0 is negative, lane 2 is positive."""
    return (lane_index - 1) * LANE_WIDTH


## 1. Lanelet-Style Map & A* Global Route

In [ ]:

class LaneletMap:
    """Straight highway lanelet map. Nodes are (lane_index, segment_index)."""
    def __init__(self, length=500.0, segment_length=10.0, n_lanes=3):
        self.length = length
        self.segment_length = segment_length
        self.n_lanes = n_lanes
        self.n_segments = int(length // segment_length)
        self.segment_centers = np.arange(self.n_segments) * segment_length + segment_length / 2.0

        self.G = nx.DiGraph()
        for lane in range(n_lanes):
            for seg in range(self.n_segments):
                self.G.add_node((lane, seg))

        for lane in range(n_lanes):
            for seg in range(self.n_segments):
                if seg < self.n_segments - 1:
                    # same-lane forward
                    self.G.add_edge((lane, seg), (lane, seg+1), cost=1.0)
                    # lane change edges
                    if lane > 0:
                        self.G.add_edge((lane, seg), (lane-1, seg+1), cost=1.5)
                    if lane < n_lanes - 1:
                        self.G.add_edge((lane, seg), (lane+1, seg+1), cost=1.5)

    def nearest_node(self, lane, s):
        seg_idx = int(np.clip(s // self.segment_length, 0, self.n_segments - 1))
        return (lane, seg_idx)

    def node_to_xy(self, node):
        lane, seg = node
        x = self.segment_centers[seg]
        y = lane_center_y(lane)
        return x, y


In [ ]:

def lanelet_heuristic(n1, n2):
    (lane1, seg1) = n1
    (lane2, seg2) = n2
    return abs(seg2 - seg1) + 0.5 * abs(lane2 - lane1)

def lanelet_astar(lmap: LaneletMap, start_node, goal_node):
    G = lmap.G
    import heapq

    open_set = []
    heapq.heappush(open_set, (0.0, start_node))
    came_from = {}
    g_score = {start_node: 0.0}

    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal_node:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start_node)
            return path[::-1]

        for neighbor in G.successors(current):
            cost = G.edges[current, neighbor]['cost']
            tentative_g = g_score[current] + cost
            if tentative_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                f = tentative_g + lanelet_heuristic(neighbor, goal_node)
                heapq.heappush(open_set, (f, neighbor))
    return None


In [ ]:

def build_route_lane_profile(lmap: LaneletMap, global_path_nodes):
    """Create desired_lane(s) from lanelet A* path."""
    if not global_path_nodes:
        return None, None

    seg_to_lane = {}
    for lane, seg in global_path_nodes:
        seg_to_lane[seg] = lane

    s_list = []
    lane_list = []
    for seg, lane in seg_to_lane.items():
        s_center = lmap.segment_centers[seg]
        s_list.append(s_center)
        lane_list.append(lane)

    s_array = np.array(s_list)
    lane_array = np.array(lane_list)

    def desired_lane_fn(s):
        seg_idx = int(np.clip(s // lmap.segment_length, 0, lmap.n_segments - 1))
        idx = np.argmin(np.abs(lmap.segment_centers - lmap.segment_centers[seg_idx]))
        return int(lane_array[np.clip(idx, 0, len(lane_array)-1)])

    return desired_lane_fn, (s_array, lane_array)


## 2. Vehicle Model
Ego uses a simplified dynamic bicycle model; others use lane-based kinematics.

In [ ]:

class Vehicle:
    def __init__(self, lane_index, s, speed, is_ego=False):
        self.lane_index = float(lane_index)   # logical lane index
        self.s = float(s)                     # Frenet s (≈ x)
        self.speed = float(speed)             # store as v_x for ego
        self.is_ego = is_ego
        # physical pose
        self.x = s
        self.y = lane_center_y(lane_index)
        self.yaw = 0.0                        # rad, 0 for straight road

        # dynamic bicycle states (ego only, but we keep on all for simplicity)
        self.vx = max(speed, 0.1)             # longitudinal velocity
        self.vy = 0.0                         # lateral velocity
        self.r  = 0.0                         # yaw rate

def vehicle_xy(v: Vehicle):
    return v.x, v.y

def create_highway_scenario(lmap: LaneletMap,
                            num_vehicles,
                            ego_lane,
                            ego_s,
                            ego_speed,
                            seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    vehicles = []
    ego = Vehicle(float(ego_lane), ego_s, ego_speed, is_ego=True)
    vehicles.append(ego)

    for _ in range(num_vehicles):
        lane = random.randint(0, lmap.n_lanes-1)
        s = random.uniform(0.0, lmap.length)
        speed = random.uniform(8.0, 30.0)
        v = Vehicle(lane, s, speed, is_ego=False)
        vehicles.append(v)

    return vehicles, ego


## 3. Visualization – Lanes, Oriented Rectangles, Lattice Candidates

In [ ]:

def draw_lane_background(lmap: LaneletMap, ax):
    for lane in range(lmap.n_lanes):
        y_center = lane_center_y(lane)
        y_bottom = y_center - LANE_WIDTH/2
        rect = Rectangle((0, y_bottom),
                         lmap.length,
                         LANE_WIDTH,
                         facecolor='lightgray',
                         alpha=0.2,
                         edgecolor='none')
        ax.add_patch(rect)
        ax.axhline(y_center, color='gray', linestyle='--', linewidth=1, alpha=0.6)


In [ ]:

def get_vehicle_corners(v: Vehicle):
    """Return 4 corners of oriented rectangle in world coords (x,y)."""
    x, y = v.x, v.y
    L, W = VEHICLE_LENGTH, VEHICLE_WIDTH
    yaw = v.yaw
    corners_local = np.array([
        [-L/2, -W/2],
        [ L/2, -W/2],
        [ L/2,  W/2],
        [-L/2,  W/2]
    ])
    c, s = np.cos(yaw), np.sin(yaw)
    R = np.array([[c, -s],
                  [s,  c]])
    rotated = (R @ corners_local.T).T
    rotated[:,0] += x
    rotated[:,1] += y
    return rotated


In [ ]:

def draw_vehicle_rect(ax, v: Vehicle):
    corners = get_vehicle_corners(v)
    x_min, x_max = corners[:,0].min(), corners[:,0].max()
    y_min, y_max = corners[:,1].min(), corners[:,1].max()
    color = '#D55E00' if v.is_ego else '#0077BB'
    ax.add_patch(Rectangle((x_min, y_min),
                           x_max - x_min,
                           y_max - y_min,
                           facecolor=color,
                           edgecolor='k',
                           alpha=0.9 if v.is_ego else 0.7))


In [ ]:

def draw_global_and_local(lmap: LaneletMap,
                          vehicles,
                          global_path_nodes,
                          ego_xy_hist,
                          ego_lane_hist,
                          local_path,
                          behavior,
                          ax_global,
                          ax_local):
    ax_global.clear()
    ax_local.clear()

    draw_lane_background(lmap, ax_global)

    # global route
    if global_path_nodes:
        xs = []; ys = []
        for node in global_path_nodes:
            x, y = lmap.node_to_xy(node)
            xs.append(x); ys.append(y)
        ax_global.plot(xs, ys, color='#0077BB', linewidth=3, label='Global route')

    # ego history
    if len(ego_xy_hist) > 1:
        xs_h = np.array([p[0] for p in ego_xy_hist])
        ys_h = np.array([p[1] for p in ego_xy_hist])
        ax_global.plot(xs_h, ys_h, color='black', linewidth=2.5, label='Ego history')

    # local trajectory & candidates
    if local_path is not None:
        x_l = local_path['x']
        y_best = local_path['y']
        candidates = local_path.get('candidates', None)
        best_idx = local_path.get('best_index', None)

        if candidates is not None:
            for i, cand in enumerate(candidates):
                y_c = cand['d']
                x_c = cand.get('x', x_l)
                if best_idx is not None and i == best_idx:
                    continue
                n = min(len(x_c), len(y_c))
                ax_global.plot(x_c[:n], y_c[:n],
                               color='gray', linestyle='--', linewidth=1.0, alpha=0.6)

        ax_global.plot(x_l, y_best, color='#009E73', linestyle='--', linewidth=2, label='Local traj')

    # vehicles
    for v in vehicles:
        draw_vehicle_rect(ax_global, v)

    if behavior is not None:
        ax_global.text(0.02, 0.95, f"Behavior: {behavior}",
                       transform=ax_global.transAxes,
                       fontsize=11, va='top', ha='left',
                       bbox=dict(boxstyle='round,pad=0.3',
                                 facecolor='white', alpha=0.8))

    ax_global.set_xlim(0, lmap.length)
    ax_global.set_ylim(-2*LANE_WIDTH, (lmap.n_lanes-1)*LANE_WIDTH + 2*LANE_WIDTH)
    ax_global.set_xlabel('x (m)')
    ax_global.set_ylabel('y (m)')
    ax_global.set_title('Global View: Route, Ego, Traffic, Local Trajectories')
    ax_global.grid(True, alpha=0.3)
    ax_global.legend(loc='lower right', fontsize=8)

    # Local zoom
    if local_path is not None:
        x_l = local_path['x']
        y_best = local_path['y']
        candidates = local_path.get('candidates', None)
        best_idx = local_path.get('best_index', None)

        if candidates is not None:
            for i, cand in enumerate(candidates):
                y_c = cand['d']
                x_c = cand.get('x', x_l)
                if best_idx is not None and i == best_idx:
                    continue
                n = min(len(x_c), len(y_c))
                ax_local.plot(x_c[:n], y_c[:n],
                              color='gray', linestyle='--', linewidth=1.0, alpha=0.6, label='_cand')

        ax_local.plot(x_l, y_best, color='#009E73', linewidth=3, label='Local traj')
        ax_local.scatter(x_l[0], y_best[0], c='#D55E00', s=60, label='Start')

        x0 = x_l[0]
        xs_local = np.linspace(x0-10, x0+60, 100)
        for lane in range(lmap.n_lanes):
            y_c = lane_center_y(lane)
            ax_local.plot(xs_local, np.ones_like(xs_local)*y_c,
                          '--', color='gray', alpha=0.5, linewidth=1)

        ax_local.set_xlabel('x (m)')
        ax_local.set_ylabel('y (m)')
        ax_local.set_title('Local Planned Trajectories (Zoomed)')
        ax_local.grid(True, alpha=0.3)
        ax_local.legend(fontsize=8)

        ax_local.set_xlim(x0-5, x0+60)
        y0 = y_best[0]
        ax_local.set_ylim(y0 - 2*LANE_WIDTH, y0 + 2*LANE_WIDTH)
        ax_local.set_aspect('equal','box')


## 4. Oriented Rectangle Collision via Separating Axis Theorem (SAT)

In [ ]:

def _project_polygon(axis, corners):
    dots = corners @ axis
    return np.min(dots), np.max(dots)

def rects_intersect(v1: Vehicle, v2: Vehicle):
    c1 = get_vehicle_corners(v1)
    c2 = get_vehicle_corners(v2)

    axes = []

    # axes from v1 edges
    for i in range(2):
        edge = c1[(i+1)%4] - c1[i]
        n = np.array([edge[0], edge[1]])
        if np.linalg.norm(n) < 1e-6:
            continue
        n = n / np.linalg.norm(n)
        axes.append(n)

    # axes from v2 edges
    for i in range(2):
        edge = c2[(i+1)%4] - c2[i]
        n = np.array([edge[0], edge[1]])
        if np.linalg.norm(n) < 1e-6:
            continue
        n = n / np.linalg.norm(n)
        axes.append(n)

    for axis in axes:
        min1, max1 = _project_polygon(axis, c1)
        min2, max2 = _project_polygon(axis, c2)
        if max1 < min2 or max2 < min1:
            return False
    return True


## 5. Behavior FSM with Lane-Change Lock (Route-Aware)

In [ ]:

BEHAVIOR_STATES = [
    'CRUISE',
    'FOLLOW',
    'STOP',
    'LANE_CHANGE_LEFT',
    'LANE_CHANGE_RIGHT',
    'ACCELERATE'
]

base_transition_cost = {s: {} for s in BEHAVIOR_STATES}
def set_cost(s_from, s_to, cost):
    base_transition_cost[s_from][s_to] = cost

for s in BEHAVIOR_STATES:
    set_cost(s, s, 1.0)

set_cost('CRUISE', 'FOLLOW', 2.0)
set_cost('CRUISE', 'ACCELERATE', 2.0)
set_cost('CRUISE', 'LANE_CHANGE_LEFT', 3.0)
set_cost('CRUISE', 'LANE_CHANGE_RIGHT', 3.0)
set_cost('CRUISE', 'STOP', 5.0)

set_cost('FOLLOW', 'CRUISE', 2.0)
set_cost('FOLLOW', 'LANE_CHANGE_LEFT', 2.5)
set_cost('FOLLOW', 'LANE_CHANGE_RIGHT', 2.5)
set_cost('FOLLOW', 'STOP', 4.0)

set_cost('STOP', 'CRUISE', 2.0)
set_cost('STOP', 'ACCELERATE', 1.5)

set_cost('LANE_CHANGE_LEFT', 'CRUISE', 2.0)
set_cost('LANE_CHANGE_RIGHT', 'CRUISE', 2.0)

set_cost('ACCELERATE', 'CRUISE', 1.5)
set_cost('ACCELERATE', 'FOLLOW', 3.0)


In [ ]:

def compute_distances_to_vehicles(vehicles, ego: Vehicle, max_dist=1e6):
    d_same = max_dist
    d_left = max_dist
    d_right = max_dist
    lead_speed = None
    ego_lane_disc = int(round(ego.lane_index))

    for v in vehicles:
        if v.is_ego:
            continue
        if v.x > ego.x:
            gap = v.x - ego.x
            lane_v = int(round(v.lane_index))
            if lane_v == ego_lane_disc and gap < d_same:
                d_same = gap
                lead_speed = v.speed
            if lane_v == ego_lane_disc - 1 and gap < d_left:
                d_left = gap
            if lane_v == ego_lane_disc + 1 and gap < d_right:
                d_right = gap
    return d_same, d_left, d_right, lead_speed


In [ ]:

SAFE_FOLLOW_DIST = 25.0
SAFE_LANECHANGE_DIST = 30.0

def traffic_penalty(current, nxt, d_same, d_left, d_right, lane_index, desired_lane):
    penalty = 0.0
    if d_same < 5.0 and nxt != 'STOP':
        penalty += 50.0

    if nxt == 'FOLLOW':
        if d_same < SAFE_FOLLOW_DIST:
            penalty += 0.0
        else:
            penalty += 5.0

    if nxt == 'CRUISE' and d_same < SAFE_FOLLOW_DIST:
        penalty += 10.0

    if nxt == 'ACCELERATE' and d_same < SAFE_FOLLOW_DIST:
        penalty += 20.0

    if nxt == 'LANE_CHANGE_LEFT':
        if lane_index == 0:
            penalty += 100.0
        if d_left < SAFE_LANECHANGE_DIST:
            penalty += 30.0
        if lane_index > desired_lane:
            penalty -= 3.0

    if nxt == 'LANE_CHANGE_RIGHT':
        if lane_index == 2:
            penalty += 100.0
        if d_right < SAFE_LANECHANGE_DIST:
            penalty += 30.0
        if lane_index < desired_lane:
            penalty -= 3.0

    if nxt != current:
        penalty += 0.5

    return penalty


In [ ]:

class SM:
    def start(self):
        self.state = self.startstate
    def step(self, inp):
        (ns, out) = self.getNextValue(self.state, inp)
        self.state = ns
        return out

class BehaviorSM(SM):
    def __init__(self):
        self.startstate = 'CRUISE'
        self.lane_change_active = False
        self.current_lc_target_lane = None

    def getNextValue(self, state, inp):
        d_same = inp['d_same']
        d_left = inp['d_left']
        d_right = inp['d_right']
        lane = inp['lane']
        desired_lane = inp['desired_lane']
        lc_complete = inp.get('lc_complete', False)

        if self.lane_change_active and state in ['LANE_CHANGE_LEFT', 'LANE_CHANGE_RIGHT']:
            if lc_complete:
                self.lane_change_active = False
                self.current_lc_target_lane = None
                return 'CRUISE', 'CRUISE'
            else:
                return state, state

        best_state = state
        best_cost = float('inf')
        for nxt in BEHAVIOR_STATES:
            base = base_transition_cost.get(state, {}).get(nxt, 10.0)
            pen = traffic_penalty(state, nxt, d_same, d_left, d_right, lane, desired_lane)
            cost = base + pen
            if cost < best_cost:
                best_cost = cost
                best_state = nxt

        if best_state == 'LANE_CHANGE_LEFT':
            self.lane_change_active = True
            self.current_lc_target_lane = max(0, lane-1)
        elif best_state == 'LANE_CHANGE_RIGHT':
            self.lane_change_active = True
            self.current_lc_target_lane = min(2, lane+1)

        return best_state, best_state


## 6. IDM Longitudinal Model (Ego & Others)

In [ ]:

def idm_acceleration(v, s_gap, v_lead,
                     v0=25.0,
                     a_max=1.5,
                     b=3.0,
                     s0=2.0,
                     T=1.5,
                     delta=4.0):
    v = max(v, 0.0)
    if s_gap <= 0.0:
        return -b
    free_term = (v / max(v0, 0.1))**delta
    if v_lead is None or np.isinf(s_gap):
        interaction = 0.0
    else:
        dv = v - v_lead
        s_star = s0 + v*T + (v*dv)/(2.0*np.sqrt(a_max*b) + 1e-6)
        s_star = max(s0, s_star)
        interaction = (s_star / s_gap)**2
    a = a_max * (1.0 - free_term - interaction)
    return a


In [ ]:

def behavior_desired_speed(behavior):
    if behavior == 'CRUISE':
        return 22.0
    elif behavior == 'FOLLOW':
        return 18.0
    elif behavior == 'STOP':
        return 0.0
    elif behavior == 'ACCELERATE':
        return 28.0
    elif behavior in ['LANE_CHANGE_LEFT', 'LANE_CHANGE_RIGHT']:
        return 20.0
    else:
        return 20.0


### 6.1 Dynamic Bicycle Update for Ego (with Stanley-style Steering)

In [ ]:

# Dynamic bicycle parameters (simplified)
m  = 1500.0   # mass [kg]
Iz = 2500.0   # yaw inertia [kg m^2]
lf = 1.2     # distance from CG to front axle [m]
lr = 1.6     # distance from CG to rear axle [m]
Cf = 80000.0  # cornering stiffness front [N/rad]
Cr = 80000.0  # cornering stiffness rear [N/rad]
L = lf + lr

def stanley_steer(ego: Vehicle, target_lane):
    """Stanley-like steering to drive ego toward the center of target lane.
    For straight road, reference heading is 0 and reference y is lane_center_y(target_lane).
    """
    y_ref = lane_center_y(target_lane)
    e_y = y_ref - ego.y               # lateral error (positive if ego below ref)
    e_psi = - ego.yaw                 # heading error (ref heading = 0)

    k = 1.5
    v = max(ego.vx, 0.1)
    delta = e_psi + np.arctan2(k * e_y, v)
    max_steer = np.deg2rad(25.0)
    delta = np.clip(delta, -max_steer, max_steer)
    return delta

def update_ego_dynamics(ego: Vehicle, behavior, dt, lead_speed, d_same, target_lane):
    """Update ego using a simplified dynamic bicycle model.

    States: x, y, yaw, vx, vy, r
    Controls:
      - a_x from IDM (behavior-based desired speed)
      - δ from Stanley-like steering toward target lane center
    """
    v0 = behavior_desired_speed(behavior)
    if behavior == 'STOP':
        a_long = -4.0
    else:
        s_gap = d_same if d_same < 1e6 else np.inf
        a_long = idm_acceleration(ego.vx, s_gap, lead_speed, v0=v0)
    a_long = np.clip(a_long, -4.0, 2.0)

    # steering from Stanley toward target lane center
    delta = stanley_steer(ego, target_lane)

    vx = max(ego.vx, 0.1)
    vy = ego.vy
    r  = ego.r
    psi = ego.yaw

    # linear tire model, small angles
    alpha_f = delta - (vy + lf * r) / vx
    alpha_r = - (vy - lr * r) / vx

    Fyf = Cf * alpha_f
    Fyr = Cr * alpha_r

    # vehicle-body frame accelerations
    ax = a_long - (Fyf * np.sin(delta)) / m + vy * r  # approx
    ay = (Fyf * np.cos(delta) + Fyr) / m - vx * r

    # yaw rate dynamics
    r_dot = (lf * Fyf * np.cos(delta) - lr * Fyr) / Iz

    # integrate velocities
    vx = vx + ax * dt
    vy = vy + ay * dt
    vx = np.clip(vx, 0.0, 40.0)

    # integrate yaw
    r = r + r_dot * dt
    psi = psi + r * dt

    # integrate position in world frame
    x_dot = vx * np.cos(psi) - vy * np.sin(psi)
    y_dot = vx * np.sin(psi) + vy * np.cos(psi)
    x = ego.x + x_dot * dt
    y = ego.y + y_dot * dt

    ego.x = x
    ego.y = y
    ego.s = ego.x
    ego.vx = vx
    ego.vy = vy
    ego.r  = r
    ego.yaw = psi
    ego.speed = vx    # keep speed alias for logging


### 6.2 Other Vehicles Physics (IDM Kinematic)

In [ ]:

def update_other_vehicles_physics(vehicles, dt):
    """Update non-ego vehicles with IDM per lane and prevent overlap."""
    lanes = {}
    for v in vehicles:
        if v.is_ego:
            continue
        lane_disc = int(round(v.lane_index))
        lanes.setdefault(lane_disc, []).append(v)

    for lane, vs in lanes.items():
        vs.sort(key=lambda v: v.x)
        for i, v in enumerate(vs):
            if i == len(vs) - 1:
                s_gap = np.inf
                v_lead = None
            else:
                lead = vs[i+1]
                s_gap = max(lead.x - v.x - VEHICLE_LENGTH, 0.1)
                v_lead = lead.speed
            a = idm_acceleration(v.speed, s_gap, v_lead, v0=25.0)
            a = np.clip(a, -4.0, 2.0)
            dv = np.clip(a * dt, -3.0 * dt, 3.0 * dt)
            v.speed = max(0.0, v.speed + dv)
            v.x += v.speed * dt
            v.s = v.x
            v.y = lane_center_y(v.lane_index)
            v.yaw = 0.0

        vs.sort(key=lambda v: v.x)
        for i in range(len(vs)-1):
            lead = vs[i+1]
            foll = vs[i]
            if lead.x - foll.x < VEHICLE_LENGTH:
                foll.x = lead.x - VEHICLE_LENGTH
                foll.s = foll.x
                foll.speed = min(foll.speed, lead.speed)


## 7. Frenet Lattice – Lateral (Multi-Lane, Multi-Horizon)

In [ ]:

def quintic_coeffs(d0, v0, a0, dT, vT, aT, T):
    A = np.array([
        [0,    0,    0,    0,    0,   1],
        [T**5, T**4, T**3, T**2, T,   1],
        [0,    0,    0,    0,    1,   0],
        [5*T**4, 4*T**3, 3*T**2, 2*T, 1, 0],
        [0,    0,    0,    2,    0,   0],
        [20*T**3, 12*T**2, 6*T, 2,   0, 0]
    ], dtype=float)
    b = np.array([d0, dT, v0, vT, a0, aT], dtype=float)
    coeffs = np.linalg.solve(A, b)
    return coeffs


In [ ]:

def sample_lattice_candidates(ego: Vehicle, behavior, route_desired_lane, lmap: LaneletMap):
    current_lane_disc = int(round(ego.lane_index))
    route_lane_now = route_desired_lane(ego.s)

    lanes = set([current_lane_disc, route_lane_now])
    if behavior == 'LANE_CHANGE_LEFT' and current_lane_disc > 0:
        lanes.add(current_lane_disc - 1)
    if behavior == 'LANE_CHANGE_RIGHT' and current_lane_disc < lmap.n_lanes-1:
        lanes.add(current_lane_disc + 1)

    horizons = [3.0, 4.0, 5.0]
    candidates = []
    for ln in sorted([max(0, min(lmap.n_lanes-1, ln)) for ln in lanes]):
        for T in horizons:
            candidates.append((ln, T))
    return candidates


In [ ]:

def evaluate_lattice_trajectory(d_traj, t, ego: Vehicle, vehicles, lmap: LaneletMap):
    dt = t[1] - t[0] if len(t) > 1 else 0.1
    v_lat = np.gradient(d_traj, dt)
    a_lat = np.gradient(v_lat, dt)
    j_lat = np.gradient(a_lat, dt)
    jerk_cost = np.sum(j_lat**2) * dt

    s_traj = ego.x + ego.vx * t
    min_dist = 1e6
    for v_other in vehicles:
        if v_other.is_ego:
            continue
        s_other = v_other.x + v_other.speed * t
        d_other = lane_center_y(v_other.lane_index)
        dist = np.sqrt((s_traj - s_other)**2 + (d_traj - d_other)**2)
        min_dist = min(min_dist, float(np.min(dist)))
    collision_cost = 0.0
    if min_dist < 5.0:
        collision_cost += 1e6
    elif min_dist < 15.0:
        collision_cost += (15.0 - min_dist)*100.0

    return jerk_cost + collision_cost


In [ ]:
def compute_frenet_lattice_local(ego: Vehicle,
                                 behavior,
                                 vehicles,
                                 lmap: LaneletMap,
                                 route_desired_lane,
                                 horizon_max=5.0,
                                 dt=0.2):
    """
    Frenet lattice planner (lateral) with decoupled longitudinal profile.
    - For each candidate (lane_T, T) we generate:
        * t: 0..T with step dt
        * d_traj: quintic lateral polynomial
        * x_traj: ego.x + ego.vx * t  (same grid, decoupled from IDM)
    - We store per-candidate x, t, d so plotting never mismatches.
    """
    cand_specs = sample_lattice_candidates(ego, behavior, route_desired_lane, lmap)

    d0 = lane_center_y(ego.lane_index)
    v0 = 0.0
    a0 = 0.0

    best_cost = float('inf')
    best_d = None
    best_lane = int(round(ego.lane_index))
    best_idx = None
    best_T = 4.0
    best_t = None
    candidates = []

    vx0 = max(ego.vx, 0.1)

    for idx, (lane_T, T) in enumerate(cand_specs):
        t = np.arange(0.0, T + 1e-6, dt)
        dT = lane_center_y(lane_T)
        vT = 0.0
        aT = 0.0
        a5, a4, a3, a2, a1, a0c = quintic_coeffs(d0, v0, a0, dT, vT, aT, T)
        d_traj = a5*t**5 + a4*t**4 + a3*t**3 + a2*t**2 + a1*t + a0c

        # consistent longitudinal profile on same time grid
        x_traj_cand = ego.x + vx0 * t

        candidates.append({
            'lane': lane_T,
            'T': T,
            'd': d_traj,
            't': t,
            'x': x_traj_cand,
        })

        cost = evaluate_lattice_trajectory(d_traj, t, ego, vehicles, lmap)
        route_lane_now = route_desired_lane(ego.s)
        cost += abs(lane_T - route_lane_now)*50.0
        cost += (T-4.0)**2 * 5.0

        if cost < best_cost:
            best_cost = cost
            best_d = d_traj
            best_lane = lane_T
            best_idx = idx
            best_T = T
            best_t = t

    # best candidate longitudinal trajectory (same grid as best_d)
    t = best_t
    x_traj = ego.x + vx0 * t

    return {
        'x': x_traj,
        'y': best_d,
        't': t,
        'target_lane': best_lane,
        'candidates': candidates,
        'best_index': best_idx
    }


## 8. Full Simulation Loop (Dynamic Ego, SAT Collision, FSM, IDM, Lattice)

In [ ]:

def simulate_full_run(road_length,
                       num_vehicles,
                       ego_s_start,
                       ego_lane_start,
                       ego_s_goal,
                       ego_lane_goal,
                       ego_speed,
                       random_seed=0,
                       sim_dt=0.1,
                       sim_Tmax=40.0):

    lmap = LaneletMap(length=road_length, segment_length=10.0, n_lanes=3)

    start_node = lmap.nearest_node(int(ego_lane_start), ego_s_start)
    goal_node = lmap.nearest_node(int(ego_lane_goal), ego_s_goal)
    global_path_nodes = lanelet_astar(lmap, start_node, goal_node)

    vehicles, ego = create_highway_scenario(
        lmap, num_vehicles, int(ego_lane_start), ego_s_start, ego_speed, seed=random_seed
    )

    route_desired_lane_fn, route_profile = build_route_lane_profile(lmap, global_path_nodes)

    bsm = BehaviorSM()
    bsm.start()

    times = []
    ego_xy_hist = []
    ego_lane_hist = []
    ego_speed_hist = []
    behavior_hist = []

    frames = []
    t = 0.0
    step = 0
    max_steps = int(sim_Tmax / sim_dt)

    while step < max_steps and ego.x < ego_s_goal:
        update_other_vehicles_physics(vehicles, sim_dt)

        d_same, d_left, d_right, lead_speed = compute_distances_to_vehicles(vehicles, ego)

        lc_complete = False
        if bsm.lane_change_active and bsm.current_lc_target_lane is not None:
            if abs(ego.lane_index - bsm.current_lc_target_lane) < 0.05:
                ego.lane_index = float(bsm.current_lc_target_lane)
                lc_complete = True

        desired_lane_now = route_desired_lane_fn(ego.s) if route_desired_lane_fn is not None else int(round(ego.lane_index))

        inp = {
            'd_same': d_same,
            'd_left': d_left,
            'd_right': d_right,
            'lane': int(round(ego.lane_index)),
            'speed': ego.speed,
            'desired_lane': desired_lane_now,
            'lc_complete': lc_complete
        }
        behavior = bsm.step(inp)

        local_path = compute_frenet_lattice_local(
            ego, behavior, vehicles, lmap, route_desired_lane_fn,
            horizon_max=5.0, dt=0.2
        )

        # Which lane should controllers track?
        if bsm.current_lc_target_lane is not None:
            target_lane = bsm.current_lc_target_lane
        else:
            target_lane = local_path['target_lane']

        # Dynamic bicycle update for ego
        update_ego_dynamics(ego, behavior, sim_dt, lead_speed, d_same, target_lane)

        # Lane index as continuous variable for planner logic
        # For straight road, lane index ~ inverse mapping of y
        ego.lane_index = np.clip(1 + ego.y / LANE_WIDTH, 0.0, 2.0)

        # Oriented rectangle collision detection
        for v in vehicles:
            if v.is_ego:
                continue
            if rects_intersect(ego, v):
                ego.vx = 0.0
                ego.speed = 0.0
                v.speed = 0.0

        times.append(t)
        ego_xy_hist.append((ego.x, ego.y))
        ego_lane_hist.append(ego.lane_index)
        ego_speed_hist.append(ego.speed)
        behavior_hist.append(behavior)

        if step % 2 == 0:
            fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(18,6))
            draw_global_and_local(lmap, vehicles, global_path_nodes,
                                  ego_xy_hist, ego_lane_hist,
                                  local_path, behavior, ax0, ax1)
            fig.tight_layout()
            fig.canvas.draw()
            img = np.asarray(fig.canvas.renderer.buffer_rgba())
            frames.append(Image.fromarray(img))
            plt.close(fig)

        t += sim_dt
        step += 1

    return {
        'lmap': lmap,
        'global_path_nodes': global_path_nodes,
        'vehicles': vehicles,
        'times': np.array(times),
        'ego_xy': np.array(ego_xy_hist),
        'ego_lane': np.array(ego_lane_hist),
        'ego_speed': np.array(ego_speed_hist),
        'behavior': behavior_hist,
        'frames': frames,
        'start_node': start_node,
        'goal_node': goal_node
    }


## 9. GIF & CSV Export

In [ ]:

def build_gif(frames, fps=10):
    tmp_dir = tempfile.mkdtemp()
    gif_path = os.path.join(tmp_dir, 'highway_sim_dynbicycle_rectcoll.gif')
    if frames:
        frames[0].save(
            gif_path,
            save_all=True,
            append_images=frames[1:],
            duration=int(1000 / fps),
            loop=0
        )
    return gif_path

def export_trajectory_csv(times, ego_xy, ego_lane, ego_speed, behaviors):
    tmp_dir = tempfile.mkdtemp()
    csv_path = os.path.join(tmp_dir, 'ego_trajectory.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['t', 'x', 'y', 'lane_index', 'speed', 'behavior'])
        for t, (x, y), lane, v, b in zip(times, ego_xy, ego_lane, ego_speed, behaviors):
            writer.writerow([float(t), float(x), float(y), float(lane), float(v), b])
    return csv_path


## 10. Gradio App

In [ ]:

def gradio_run(road_length,
               num_vehicles,
               ego_s_start,
               ego_lane_start,
               ego_s_goal,
               ego_lane_goal,
               ego_speed,
               random_seed):
    sim = simulate_full_run(
        road_length=road_length,
        num_vehicles=int(num_vehicles),
        ego_s_start=ego_s_start,
        ego_lane_start=int(ego_lane_start),
        ego_s_goal=ego_s_goal,
        ego_lane_goal=int(ego_lane_goal),
        ego_speed=ego_speed,
        random_seed=int(random_seed),
        sim_dt=0.1,
        sim_Tmax=50.0
    )

    gif_path = build_gif(sim['frames'], fps=10)
    csv_path = export_trajectory_csv(
        sim['times'], sim['ego_xy'], sim['ego_lane'], sim['ego_speed'], sim['behavior']
    )

    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(sim['times'], sim['ego_speed'])
    ax.set_xlabel('t (s)')
    ax.set_ylabel('Ego speed (m/s)')
    ax.set_title('Ego Speed vs Time')
    ax.grid(True)
    fig.tight_layout()

    debug_info = {
        'start_node': sim['start_node'],
        'goal_node': sim['goal_node'],
        'global_path_len': len(sim['global_path_nodes']) if sim['global_path_nodes'] is not None else 0,
        'final_x': float(sim['ego_xy'][-1,0]) if len(sim['ego_xy'])>0 else None,
        'final_y': float(sim['ego_xy'][-1,1]) if len(sim['ego_xy'])>0 else None,
        'final_lane_index': float(sim['ego_lane'][-1]) if len(sim['ego_lane'])>0 else None,
        'final_speed': float(sim['ego_speed'][-1]) if len(sim['ego_speed'])>0 else None,
        'steps': len(sim['times'])
    }

    return gif_path, fig, csv_path, debug_info


In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("## 🚗 Highway Stack – Dynamic Bicycle, Lanelet A*, FSM, IDM, Frenet Lattice & Oriented Rect Collision")

    with gr.Row():
        road_length = gr.Slider(100, 1000, value=500, step=10, label="Road length (m)")
        num_vehicles = gr.Slider(0, 50, value=20, step=1, label="Number of other vehicles")
        random_seed = gr.Slider(0, 100, value=0, step=1, label="Random seed")

    with gr.Row():
        ego_s_start = gr.Slider(0, 900, value=50, step=5, label="Ego start s / x (m)")
        ego_lane_start = gr.Dropdown(choices=[0,1,2], value=1, label="Ego start lane")
    with gr.Row():
        ego_s_goal = gr.Slider(0, 900, value=400, step=5, label="Ego goal s / x (m)")
        ego_lane_goal = gr.Dropdown(choices=[0,1,2], value=2, label="Ego goal lane")

    ego_speed = gr.Slider(0, 40, value=20, step=1, label="Ego initial speed (m/s)")

    run_button = gr.Button("Run Simulation")

    gif_out = gr.Image(type="filepath", label="Simulation Animation (GIF)")
    vel_plot = gr.Plot(label="Ego Speed vs Time")
    traj_file = gr.File(label="Download Ego Trajectory CSV")
    debug_json = gr.JSON(label="Debug info")

    run_button.click(
        fn=gradio_run,
        inputs=[road_length, num_vehicles, ego_s_start, ego_lane_start,
                ego_s_goal, ego_lane_goal, ego_speed, random_seed],
        outputs=[gif_out, vel_plot, traj_file, debug_json]
    )

# In Colab / Jupyter:
# 1. Run all cells
demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://187746a2ada9fb5cc3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
